# Orbit Wars AI Training on Google Colab 🚀

**Complete training pipeline using free GPU**

- ✅ Phase 2: Behavioral Cloning (30 mins)
- ✅ Phase 3: PPO Training (2-4 hours)
- ✅ Phase 4: Evaluation & Deployment

**GPU:** NVIDIA A100 or T4 (10-50x faster than CPU)

**RAM:** 16GB (2x your laptop)

## Step 1: Setup Environment & Mount Google Drive

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
import os
import sys

drive.mount('/content/gdrive')
print("✅ Google Drive mounted")

In [ ]:
# Option A: If code is in Google Drive as ZIP
import shutil
import zipfile

# Unzip Orbit Wars project
zip_path = '/content/gdrive/My Drive/Orbit_wars.zip'  # Change if different location
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content')
    print("✅ Code extracted from Drive")
else:
    print("⚠️ ZIP not found. Using Option B (GitHub clone) instead...")

In [ ]:
# Option B: Clone from GitHub (if you pushed there)
# Uncomment below to use:
# !git clone https://github.com/YOUR_USERNAME/orbit-wars-rl.git /content/orbit-wars
# os.chdir('/content/orbit-wars/Orbit_wars')

# For now, assume code is already in /content/Orbit_wars
os.chdir('/content/Orbit_wars')
print(f"📁 Working directory: {os.getcwd()}")
print(f"📋 Files: {os.listdir('.')[:10]}")

In [ ]:
# Verify GPU
import torch

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU not detected! Go to Runtime → Change Runtime Type → GPU")

In [ ]:
# Install dependencies
!pip install -q torch numpy kaggle-environments gymnasium matplotlib pandas tensorboard -U
print("✅ Dependencies installed")

In [ ]:
# Create required directories
import os
os.makedirs('data/bc_dataset', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('logs', exist_ok=True)
os.makedirs('runs', exist_ok=True)
print("✅ Directories created")

## Step 2: Phase 2 - Behavioral Cloning (Optional)

Generate 100k expert rollouts from HeuristicBaseline to pre-train the Transformer.

In [ ]:
# If training/generate_bc_data.py exists, generate BC data
import os

if os.path.exists('training/generate_bc_data.py'):
    print("⏳ Generating BC data (100k rollouts)...")
    # This will take ~30 mins on Colab GPU
    # !python training/generate_bc_data.py --num_rollouts 100000 --save_dir data/bc_dataset --device cuda
    print("Uncomment above to run BC data generation")
else:
    print("⚠️ generate_bc_data.py not found. Skipping Phase 2.")

## Step 3: Phase 3 - PPO Training

Core RL training loop. Train agent against fixed heuristic baseline.

In [ ]:
# Verify train_ppo.py exists
if os.path.exists('training/train_ppo.py'):
    print("✅ train_ppo.py found")
    # Show first few lines
    with open('training/train_ppo.py', 'r') as f:
        lines = f.readlines()[:20]
        print('\n'.join(lines))
else:
    print("⚠️ train_ppo.py not found. Need to implement it first.")

In [ ]:
# Run PPO Training (500k steps = ~2 hours on Colab GPU)
# This assumes train_ppo.py is properly implemented

print("⏳ Starting PPO training...")
print("Expected duration: 2-4 hours")
print("\nRunning:")
print("python training/train_ppo.py --total_timesteps 500000 --opponent baseline --batch_size 32 --device cuda")

# Uncomment to run (or implement train_ppo.py first):
# !python training/train_ppo.py \
#   --total_timesteps 500000 \
#   --opponent baseline \
#   --batch_size 32 \
#   --device cuda \
#   --save_interval 50000 \
#   --log_dir logs/ppo_training

## Step 4: Monitor Training with TensorBoard

In [ ]:
# If training logs exist, launch TensorBoard
import os

if os.path.exists('logs/ppo_training'):
    %load_ext tensorboard
    %tensorboard --logdir logs/ppo_training
else:
    print("⚠️ Training logs not found. Run PPO training first.")

## Step 5: Evaluate Agent Performance

In [ ]:
# Evaluate the trained agent against heuristic baselines
print("⏳ Running evaluation...")
print("This will play 50 games against the heuristic baseline")

# Uncomment to run (assumes training completed):
# !python training/evaluation.py \
#   --agent_path checkpoints/ppo_500k.pt \
#   --num_games 50 \
#   --save_results logs/eval_results.txt

## Step 6: Save Results to Google Drive

In [ ]:
# Save checkpoints and logs to Google Drive (so they persist after Colab session ends)
import shutil
import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f'/content/gdrive/My Drive/orbit_wars_results_{timestamp}'
os.makedirs(save_dir, exist_ok=True)

# Copy important files
files_to_save = [
    'checkpoints/ppo_500k.pt',
    'logs/eval_results.txt',
    'logs/ppo_training/',
    'PROGRESS.md'
]

for file in files_to_save:
    if os.path.exists(file):
        if os.path.isdir(file):
            shutil.copytree(file, f'{save_dir}/{os.path.basename(file)}')
        else:
            shutil.copy(file, save_dir)
        print(f"✅ Saved: {file}")
    else:
        print(f"⚠️ Skipped (not found): {file}")

print(f"\n✅ All results saved to: {save_dir}")

## Step 7: Download Model for Local Deployment

In [ ]:
from google.colab import files
import os

# Download trained model to your machine
if os.path.exists('checkpoints/ppo_500k.pt'):
    print("📥 Downloading trained model...")
    files.download('checkpoints/ppo_500k.pt')
    print("✅ Model downloaded!")
else:
    print("⚠️ Model not found. Train it first.")

## Phase 4: Extended Training (Optional)

For multi-day training (self-play league), restart this notebook or continue below.

In [ ]:
# Continue training with 2M steps (if Phase 3 checkpoint exists)
print("⏳ Extended PPO training (Phase 4)...")
print("Expected duration: 4-6 hours")

# Uncomment to continue training:
# !python training/train_ppo.py \
#   --total_timesteps 2000000 \
#   --opponent baseline \
#   --batch_size 32 \
#   --device cuda \
#   --checkpoint_load checkpoints/ppo_500k.pt \
#   --save_interval 100000 \
#   --log_dir logs/ppo_extended

## 📊 Monitoring Checklist

- [ ] GPU is enabled and available
- [ ] Training loss is decreasing (check TensorBoard)
- [ ] Checkpoints saving to Drive every 50k steps
- [ ] Memory usage stable (<15GB)
- [ ] Session not timing out (free Colab = 12 hrs max)

## ⚠️ If Training Fails

**Out of Memory?**
```python
torch.cuda.empty_cache()  # Clear GPU memory
# Try smaller batch_size: 32 → 16
```

**Import Errors?**
```bash
pip install --upgrade kaggle-environments gymnasium
```

**Session Timed Out?**
- Restart notebook
- Load checkpoint from Drive: `checkpoints/ppo_500k.pt`
- Continue training